# Lab 05 — Algoritmos de Búsqueda
**Inteligencia Artificial 2026** | 18 de marzo de 2026

En este laboratorio simulamos y comparamos cinco métodos de búsqueda:
DFS, BFS, UCS (Dijkstra), Greedy Search y A*.

## 0. Configuración e Imports

In [ ]:
import numpy as np
import heapq
from collections import deque
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

%matplotlib inline

## 1. Clases y Funciones Base

In [ ]:
class Grid:
    """Mapa 2D con muros y pesos."""

    def __init__(self, rows, cols):
        self.rows = rows
        self.cols = cols
        self.grid = np.zeros((rows, cols), dtype=float)

    def set_wall(self, r, c):
        self.grid[r, c] = -1

    def set_weight(self, r, c, w):
        self.grid[r, c] = w

    def is_wall(self, r, c):
        return self.grid[r, c] == -1

    def cost(self, r, c):
        v = self.grid[r, c]
        return v if v > 0 else 1

    def neighbors(self, r, c):
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.rows and 0 <= nc < self.cols:
                if not self.is_wall(nr, nc):
                    yield (nr, nc)

    def add_rect_wall(self, r1, c1, r2, c2):
        for r in range(r1, r2+1):
            for c in range(c1, c2+1):
                self.set_wall(r, c)

    def add_rect_weight(self, r1, c1, r2, c2, w):
        for r in range(r1, r2+1):
            for c in range(c1, c2+1):
                self.set_weight(r, c, w)

In [ ]:
def random_maze(rows, cols, wall_prob=0.3, seed=None):
    rng = np.random.RandomState(seed)
    g = Grid(rows, cols)
    for r in range(rows):
        for c in range(cols):
            if rng.random() < wall_prob:
                g.set_wall(r, c)
    return g


def recursive_division_maze(rows, cols, seed=None):
    rng = np.random.RandomState(seed)
    g = Grid(rows, cols)

    def divide(r1, c1, r2, c2):
        h = r2 - r1
        w = c2 - c1
        if h < 2 or w < 2:
            return
        if h > w:
            wall_r = rng.randint(r1 + 1, r2)
            gap = rng.randint(c1, c2 + 1)
            for c in range(c1, c2 + 1):
                if c != gap:
                    g.set_wall(wall_r, c)
            divide(r1, c1, wall_r - 1, c2)
            divide(wall_r + 1, c1, r2, c2)
        else:
            wall_c = rng.randint(c1 + 1, c2)
            gap = rng.randint(r1, r2 + 1)
            for r in range(r1, r2 + 1):
                if r != gap:
                    g.set_wall(r, wall_c)
            divide(r1, c1, r2, wall_c - 1)
            divide(r1, wall_c + 1, r2, c2)

    divide(0, 0, rows - 1, cols - 1)
    return g

In [ ]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def euclidean(a, b):
    return ((a[0]-b[0])**2 + (a[1]-b[1])**2) ** 0.5

In [ ]:
def bfs(grid, start, goal):
    queue = deque([start])
    came_from = {start: None}
    visited_order = []
    while queue:
        node = queue.popleft()
        visited_order.append(node)
        if node == goal:
            break
        for nb in grid.neighbors(*node):
            if nb not in came_from:
                came_from[nb] = node
                queue.append(nb)
    path = reconstruct(came_from, start, goal)
    stats = {'nodos_explorados': len(visited_order), 'frontera_final': len(queue), 'longitud_camino': len(path)-1 if path else None}
    return path, visited_order, stats


def dfs(grid, start, goal):
    stack = [start]
    came_from = {start: None}
    visited = set()
    visited_order = []
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        visited_order.append(node)
        if node == goal:
            break
        for nb in grid.neighbors(*node):
            if nb not in came_from:
                came_from[nb] = node
                stack.append(nb)
    path = reconstruct(came_from, start, goal)
    stats = {'nodos_explorados': len(visited_order), 'frontera_final': len(stack), 'longitud_camino': len(path)-1 if path else None}
    return path, visited_order, stats


def ucs(grid, start, goal):
    heap = [(0, start)]
    came_from = {start: None}
    cost_so_far = {start: 0}
    visited_order = []
    while heap:
        curr_cost, node = heapq.heappop(heap)
        if curr_cost > cost_so_far.get(node, float('inf')):
            continue
        visited_order.append(node)
        if node == goal:
            break
        for nb in grid.neighbors(*node):
            new_cost = curr_cost + grid.cost(*nb)
            if new_cost < cost_so_far.get(nb, float('inf')):
                cost_so_far[nb] = new_cost
                came_from[nb] = node
                heapq.heappush(heap, (new_cost, nb))
    path = reconstruct(came_from, start, goal)
    stats = {'nodos_explorados': len(visited_order), 'frontera_final': len(heap), 'longitud_camino': len(path)-1 if path else None, 'costo_total': cost_so_far.get(goal)}
    return path, visited_order, stats


def greedy(grid, start, goal, heuristic=manhattan):
    heap = [(heuristic(start, goal), start)]
    came_from = {start: None}
    visited = set()
    visited_order = []
    while heap:
        _, node = heapq.heappop(heap)
        if node in visited:
            continue
        visited.add(node)
        visited_order.append(node)
        if node == goal:
            break
        for nb in grid.neighbors(*node):
            if nb not in came_from:
                came_from[nb] = node
                heapq.heappush(heap, (heuristic(nb, goal), nb))
    path = reconstruct(came_from, start, goal)
    stats = {'nodos_explorados': len(visited_order), 'frontera_final': len(heap), 'longitud_camino': len(path)-1 if path else None}
    return path, visited_order, stats


def astar(grid, start, goal, heuristic=manhattan):
    heap = [(heuristic(start, goal), 0, start)]
    came_from = {start: None}
    cost_so_far = {start: 0}
    visited_order = []
    while heap:
        f, curr_cost, node = heapq.heappop(heap)
        if curr_cost > cost_so_far.get(node, float('inf')):
            continue
        visited_order.append(node)
        if node == goal:
            break
        for nb in grid.neighbors(*node):
            new_cost = curr_cost + grid.cost(*nb)
            if new_cost < cost_so_far.get(nb, float('inf')):
                cost_so_far[nb] = new_cost
                came_from[nb] = node
                heapq.heappush(heap, (new_cost + heuristic(nb, goal), new_cost, nb))
    path = reconstruct(came_from, start, goal)
    stats = {'nodos_explorados': len(visited_order), 'frontera_final': len(heap), 'longitud_camino': len(path)-1 if path else None, 'costo_total': cost_so_far.get(goal)}
    return path, visited_order, stats


def reconstruct(came_from, start, goal):
    if goal not in came_from:
        return []
    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = came_from[node]
    path.reverse()
    return path

In [ ]:
def plot_result(grid, start, goal, path, visited, title='', ax=None):
    show = ax is None
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 10 * grid.rows / grid.cols))
    rows, cols = grid.rows, grid.cols
    img = np.ones((rows, cols, 3))
    for r in range(rows):
        for c in range(cols):
            if grid.grid[r, c] > 0:
                img[r, c] = [0.85, 0.78, 0.55]
            if grid.is_wall(r, c):
                img[r, c] = [0.2, 0.2, 0.3]
    for r, c in visited:
        if (r, c) != start and (r, c) != goal:
            img[r, c] = [0.68, 0.85, 0.95]
    for r, c in path:
        if (r, c) != start and (r, c) != goal:
            img[r, c] = [1.0, 0.85, 0.2]
    img[start[0], start[1]] = [0.2, 0.8, 0.2]
    img[goal[0], goal[1]] = [0.9, 0.2, 0.2]
    ax.imshow(img, interpolation='nearest')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    if show:
        plt.tight_layout(); plt.show()

In [ ]:
def animate_search(grid, start, goal, path, visited, title='', speed=10):
    rows, cols = grid.rows, grid.cols
    fig, ax = plt.subplots(figsize=(8, 8 * rows / cols))
    base = np.ones((rows, cols, 3))
    for r in range(rows):
        for c in range(cols):
            if grid.grid[r, c] > 0:
                base[r, c] = [0.85, 0.78, 0.55]
            if grid.is_wall(r, c):
                base[r, c] = [0.2, 0.2, 0.3]
    base[start[0], start[1]] = [0.2, 0.8, 0.2]
    base[goal[0], goal[1]] = [0.9, 0.2, 0.2]
    img_data = base.copy()
    im = ax.imshow(img_data, interpolation='nearest')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    total_visit_frames = (len(visited) + speed - 1) // speed
    total_frames = total_visit_frames + 10
    def update(frame):
        nonlocal img_data
        if frame < total_visit_frames:
            s = frame * speed
            e = min(s + speed, len(visited))
            for i in range(s, e):
                r, c = visited[i]
                if (r, c) != start and (r, c) != goal:
                    img_data[r, c] = [0.68, 0.85, 0.95]
        elif frame == total_visit_frames:
            for r, c in path:
                if (r, c) != start and (r, c) != goal:
                    img_data[r, c] = [1.0, 0.85, 0.2]
        im.set_data(img_data)
        return [im]
    anim = FuncAnimation(fig, update, frames=total_frames, interval=50, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

In [ ]:
def compare_two(grid, start, goal, result1, result2, title1, title2):
    path1, vis1, stats1 = result1
    path2, vis2, stats2 = result2
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7 * grid.rows / grid.cols))
    plot_result(grid, start, goal, path1, vis1, f"{title1}\nExplorados: {stats1['nodos_explorados']} | Camino: {stats1['longitud_camino']}", ax1)
    plot_result(grid, start, goal, path2, vis2, f"{title2}\nExplorados: {stats2['nodos_explorados']} | Camino: {stats2['longitud_camino']}", ax2)
    plt.tight_layout(); plt.show()
    print(f"{'Métrica':<30} {title1:<20} {title2:<20}")
    print('=' * 70)
    for k in stats1:
        print(f"{k:<30} {str(stats1.get(k,'-')):<20} {str(stats2.get(k,'-')):<20}")

---
## Ejercicio 1: El Laberinto de BFS y DFS
### 1a. Simple Random Maze

In [ ]:
maze1 = random_maze(30, 40, wall_prob=0.25, seed=42)
start1, goal1 = (1, 1), (28, 38)
maze1.grid[start1[0], start1[1]] = 0
maze1.grid[goal1[0], goal1[1]] = 0

res_bfs = bfs(maze1, start1, goal1)
res_dfs = dfs(maze1, start1, goal1)

compare_two(maze1, start1, goal1, res_bfs, res_dfs, 'BFS', 'DFS')

In [ ]:
print('Animación BFS:')
display(animate_search(maze1, start1, goal1, *res_bfs[:2], title='BFS — Simple Random Maze', speed=15))

In [ ]:
print('Animación DFS:')
display(animate_search(maze1, start1, goal1, *res_dfs[:2], title='DFS — Simple Random Maze', speed=15))

### 1b. Recursive Division Maze

In [ ]:
maze1b = recursive_division_maze(30, 40, seed=7)
start1b, goal1b = (1, 1), (28, 38)
maze1b.grid[start1b[0], start1b[1]] = 0
maze1b.grid[goal1b[0], goal1b[1]] = 0

res_bfs_1b = bfs(maze1b, start1b, goal1b)
res_dfs_1b = dfs(maze1b, start1b, goal1b)

compare_two(maze1b, start1b, goal1b, res_bfs_1b, res_dfs_1b, 'BFS (Recursive Div)', 'DFS (Recursive Div)')

---
## Ejercicio 2: La Trampa de la Distancia

In [ ]:
trap = Grid(25, 40)
start2, goal2 = (12, 2), (12, 37)

# Muro vertical largo con abertura solo arriba
for r in range(1, 24):
    trap.set_wall(r, 20)
# Segundo muro que fuerza desvío
for r in range(0, 23):
    trap.set_wall(r, 25)

res_greedy2 = greedy(trap, start2, goal2)
res_bfs2 = bfs(trap, start2, goal2)

compare_two(trap, start2, goal2, res_greedy2, res_bfs2, 'Greedy', 'BFS')

In [ ]:
print('Animación Greedy (trampa):')
display(animate_search(trap, start2, goal2, *res_greedy2[:2], title='Greedy — Escenario Trampa', speed=5))

In [ ]:
print('Animación BFS (trampa):')
display(animate_search(trap, start2, goal2, *res_bfs2[:2], title='BFS — Escenario Trampa', speed=5))

---
## Ejercicio 3: Heurísticas y Pesos — A* vs Dijkstra

In [ ]:
grid3 = Grid(30, 40)
start3, goal3 = (2, 2), (27, 37)

grid3.add_rect_wall(5, 8, 12, 10)
grid3.add_rect_wall(8, 15, 10, 25)
grid3.add_rect_wall(15, 5, 22, 7)
grid3.add_rect_wall(18, 20, 25, 22)
grid3.add_rect_wall(3, 30, 10, 32)
grid3.add_rect_wall(20, 30, 27, 31)
grid3.add_rect_wall(12, 18, 14, 19)

res_astar3 = astar(grid3, start3, goal3)
res_ucs3 = ucs(grid3, start3, goal3)

compare_two(grid3, start3, goal3, res_astar3, res_ucs3, 'A*', 'Dijkstra (UCS)')

In [ ]:
print('Animación A*:')
display(animate_search(grid3, start3, goal3, *res_astar3[:2], title='A* — Obstáculos Dispersos', speed=10))

In [ ]:
print('Animación Dijkstra:')
display(animate_search(grid3, start3, goal3, *res_ucs3[:2], title='Dijkstra — Obstáculos Dispersos', speed=10))

---
## Ejercicio 4: El Impacto de la Métrica — Manhattan vs Euclideana

In [ ]:
res_astar_man = astar(grid3, start3, goal3, heuristic=manhattan)
res_astar_euc = astar(grid3, start3, goal3, heuristic=euclidean)

compare_two(grid3, start3, goal3, res_astar_man, res_astar_euc, 'A* Manhattan', 'A* Euclidiana')

In [ ]:
# Segundo escenario
grid4b = Grid(25, 35)
start4b, goal4b = (0, 0), (24, 34)

grid4b.add_rect_wall(5, 10, 15, 11)
grid4b.add_rect_wall(10, 20, 20, 21)
grid4b.add_rect_wall(0, 25, 8, 26)

res_man4b = astar(grid4b, start4b, goal4b, heuristic=manhattan)
res_euc4b = astar(grid4b, start4b, goal4b, heuristic=euclidean)

compare_two(grid4b, start4b, goal4b, res_man4b, res_euc4b, 'A* Manhattan', 'A* Euclidiana')

---
## Ejercicio 5: El Peor Escenario (Stress-Test)
### 5a. Callejón sin salida

In [ ]:
grid5 = Grid(30, 50)
start5, goal5 = (15, 2), (15, 47)

# Canal que parece ir al objetivo pero está cerrado al final
for c in range(10, 42):
    grid5.set_wall(12, c)
    grid5.set_wall(18, c)
for r in range(12, 19):
    grid5.set_wall(r, 42)

res_greedy5 = greedy(grid5, start5, goal5)
res_astar5 = astar(grid5, start5, goal5)

compare_two(grid5, start5, goal5, res_greedy5, res_astar5, 'Greedy', 'A*')

In [ ]:
print('Animación Greedy (callejón sin salida):')
display(animate_search(grid5, start5, goal5, *res_greedy5[:2], title='Greedy — Callejón sin Salida', speed=8))

In [ ]:
print('Animación A* (callejón sin salida):')
display(animate_search(grid5, start5, goal5, *res_astar5[:2], title='A* — Callejón sin Salida', speed=8))

### 5b. Pantano de alto costo

In [ ]:
grid5b = Grid(25, 40)
start5b, goal5b = (12, 3), (12, 36)

# Pantano vertical de costo 15
grid5b.add_rect_weight(0, 18, 24, 22, 15)

res_dijk5b = ucs(grid5b, start5b, goal5b)
res_astar5b = astar(grid5b, start5b, goal5b)

compare_two(grid5b, start5b, goal5b, res_astar5b, res_dijk5b, 'A*', 'Dijkstra')

In [ ]:
print('Animación A* (pantano):')
display(animate_search(grid5b, start5b, goal5b, *res_astar5b[:2], title='A* — Pantano', speed=8))

In [ ]:
print('Animación Dijkstra (pantano):')
display(animate_search(grid5b, start5b, goal5b, *res_dijk5b[:2], title='Dijkstra — Pantano', speed=8))